## Extração de Resultados de Licitações (PNCP)

Para cada contratação em `contratacoes14133_pe.csv` com `existeResultado == True`,
consulta o PNCP para obter os itens e o CNPJ do fornecedor vencedor.

**Fluxo:**
1. Para cada contratação → GET `/orgaos/{cnpj}/compras/{ano}/{seq}/itens`
2. Para cada item → GET `/orgaos/{cnpj}/compras/{ano}/{seq}/itens/{num}/resultados`
3. Extrai CNPJ do vencedor, valor homologado, situação

**Saída:** `data/raw/propostas_pe.csv` — uma linha por (contratação × item × resultado)

Dependência: `data/raw/contratacoes14133_pe.csv` (produzido pelo NB02)

In [8]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import time
import os

os.makedirs("data/raw", exist_ok=True)
PNCP_BASE = "https://pncp.gov.br/api/pncp/v1"
RETRY_ESPERAS = [5, 15, 30]

def normaliza_cnpj(cnpj):
    return str(cnpj).replace(".", "").replace("/", "").replace("-", "").strip().zfill(14)

print("Setup OK")

Setup OK


### 1. Carrega contratações e filtra as que têm resultado

In [ ]:
df_c14 = pd.read_csv("data/raw/contratacoes14133_pe.csv", dtype=str)
print(f"{len(df_c14)} contratações carregadas")

df_com_resultado = df_c14[df_c14["existeResultado"].str.lower() == "true"].copy()
df_com_resultado["cnpj_norm"] = df_com_resultado["orgaoEntidadeCnpj"].apply(normaliza_cnpj)

print(f"{len(df_com_resultado)} contratações com existeResultado=True")
print("\nColunas chave:")
print(df_com_resultado[["idCompra", "orgaoEntidadeCnpj", "anoCompraPncp", "sequencialCompraPncp", "existeResultado"]].head(3))

11353 contratações carregadas
10057 contratações com existeResultado=True

Colunas chave:
            idCompra orgaoEntidadeCnpj anoCompraPncp sequencialCompraPncp  \
0  38919505000012023    09791450000114          2023                    1   
1  15518005000572022    15126437000143          2022                  842   
2  15518005000022023    15126437000143          2023                   43   

  existeResultado  
0            True  
1            True  
2            True  


### 2. Debug — Inspecionar estrutura dos endpoints PNCP

Testa com a primeira contratação disponível para entender campos retornados.

In [ ]:
row = df_com_resultado.iloc[0]
cnpj = row["cnpj_norm"]
ano  = row["anoCompraPncp"]
seq  = row["sequencialCompraPncp"]

print(f"Testando: cnpj={cnpj}, ano={ano}, seq={seq}")

url_itens = f"{PNCP_BASE}/orgaos/{cnpj}/compras/{ano}/{seq}/itens?pagina=1&tamanhoPagina=10"
print(f"\nGET {url_itens}")

try:
    req = urllib.request.Request(url_itens, headers={"User-Agent": "tcc-research/1.0"})
    with urllib.request.urlopen(req, timeout=20) as resp:
        itens = json.loads(resp.read().decode())
    lista_itens = itens if isinstance(itens, list) else itens.get("data", itens)
    print(f"Itens retornados: {len(lista_itens)}")
    if lista_itens:
        print("\nCampos do item:")
        for k, v in lista_itens[0].items():
            print(f"  {k}: {str(v)[:100]}")
except Exception as e:
    print(f"Erro ao buscar itens: {e}")
    lista_itens = []

Testando: cnpj=09791450000114, ano=2023, seq=1

GET https://pncp.gov.br/api/pncp/v1/orgaos/09791450000114/compras/2023/1/itens?pagina=1&tamanhoPagina=10
Itens retornados: 2

Campos do item:
  numeroItem: 1
  descricao: Prestaçao de Serviços de Agenciamento de Viagens
  materialOuServico: S
  materialOuServicoNome: Serviço
  valorUnitarioEstimado: 40.0
  valorTotal: 2400.0
  quantidade: 60.0
  unidadeMedida: UNIDADE
  orcamentoSigiloso: False
  itemCategoriaId: 3
  itemCategoriaNome: Não se aplica
  patrimonio: None
  codigoRegistroImobiliario: None
  criterioJulgamentoId: 1
  criterioJulgamentoNome: Menor preço
  situacaoCompraItem: 1
  situacaoCompraItemNome: Em andamento
  tipoBeneficio: 5
  tipoBeneficioNome: Não se aplica
  incentivoProdutivoBasico: False
  dataInclusao: 2023-01-12T07:01:54
  dataAtualizacao: 2023-01-12T07:01:54
  temResultado: True
  imagem: 0
  aplicabilidadeMargemPreferenciaNormal: False
  aplicabilidadeMargemPreferenciaAdicional: False
  percentualMargemPrefere

In [ ]:
if lista_itens:
    num_item = lista_itens[0].get("numeroItem", 1)
    url_res = f"{PNCP_BASE}/orgaos/{cnpj}/compras/{ano}/{seq}/itens/{num_item}/resultados?pagina=1&tamanhoPagina=10"
    print(f"GET {url_res}")

    try:
        req = urllib.request.Request(url_res, headers={"User-Agent": "tcc-research/1.0"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            resultados = json.loads(resp.read().decode())
        lista_res = resultados if isinstance(resultados, list) else resultados.get("data", resultados)
        print(f"Resultados retornados: {len(lista_res)}")
        if lista_res:
            print("\nCampos do resultado:")
            for k, v in lista_res[0].items():
                print(f"  {k}: {str(v)[:100]}")
    except Exception as e:
        print(f"Erro ao buscar resultados: {e}")

GET https://pncp.gov.br/api/pncp/v1/orgaos/09791450000114/compras/2023/1/itens/1/resultados?pagina=1&tamanhoPagina=10
Resultados retornados: 1

Campos do resultado:
  indicadorSubcontratacao: False
  dataInclusao: 2023-02-08T16:10:00
  numeroItem: 1
  niFornecedor: 07832586000108
  dataCancelamento: None
  dataAtualizacao: 2023-04-28T17:28:17
  tipoPessoa: PJ
  nomeRazaoSocialFornecedor: DF TURISMO E EVENTOS LTDA
  valorTotalHomologado: 0.6
  reservaRemanescente: {'codigo': 1, 'nome': 'Não se aplica'}
  timezoneCotacaoMoedaEstrangeira: None
  moedaEstrangeira: None
  valorNominalMoedaEstrangeira: None
  dataCotacaoMoedaEstrangeira: None
  codigoPais: BRA
  porteFornecedorId: 2
  quantidadeHomologada: 60.0
  valorUnitarioHomologado: 0.01
  percentualDesconto: 0.0
  amparoLegalMargemPreferencia: None
  amparoLegalCriterioDesempate: None
  paisOrigemProdutoServico: None
  localidadeExterior: None
  ordemClassificacaoSrp: 1
  dataResultado: 2023-02-08
  motivoCancelamento: None
  numeroCon

### 3. Funções de extração

Após validar os campos no debug acima, ajuste `CAMPO_CNPJ_FORNECEDOR` se necessário.
Valores padrão baseados na documentação do PNCP.

In [ ]:
import urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

CAMPO_CNPJ_FORN   = "niFornecedor"                  # CNPJ/CPF do vencedor
CAMPO_NOME_FORN   = "nomeRazaoSocialFornecedor"      # Razão social
CAMPO_VALOR       = "valorTotalHomologado"            # Valor homologado
CAMPO_SITUACAO    = "situacaoCompraItemResultadoNome" # Ex: 'Informado', 'Cancelado'
CAMPO_NUM_ITEM    = "numeroItem"                     # Número do item

def get_pncp(url, tentativas=None):
    if tentativas is None:
        tentativas = RETRY_ESPERAS
    for espera in tentativas + [None]:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "tcc-research/1.0"})
            with urllib.request.urlopen(req, timeout=20) as resp:
                data = json.loads(resp.read().decode())
            return data if isinstance(data, list) else data.get("data", data)
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None 
            if espera is None:
                return None
        
            time.sleep(espera if e.code == 429 else min(espera, 3))
        except Exception:
            if espera is None:
                return None
            time.sleep(min(espera, 3))

def extrair_resultados_contratacao(cnpj, ano, seq, id_compra):
    registros = []

    url_itens = f"{PNCP_BASE}/orgaos/{cnpj}/compras/{ano}/{seq}/itens?pagina=1&tamanhoPagina=500"
    itens = get_pncp(url_itens)
    if not itens:
        return registros

    for item in itens:
        num_item = item.get(CAMPO_NUM_ITEM, 1)
        url_res = f"{PNCP_BASE}/orgaos/{cnpj}/compras/{ano}/{seq}/itens/{num_item}/resultados?pagina=1&tamanhoPagina=500"
        resultados = get_pncp(url_res)

        if not resultados:
            continue
        for r in resultados:
            registros.append({
                "idCompra":               id_compra,
                "orgaoEntidadeCnpj":      cnpj,
                "anoCompraPncp":          ano,
                "sequencialCompraPncp":   seq,
                "numeroItem":             num_item,
                "descricaoItem":          item.get("descricao", ""),
                "cnpj_fornecedor":        r.get(CAMPO_CNPJ_FORN),
                "nome_fornecedor":        r.get(CAMPO_NOME_FORN),
                "valor_total_homologado": r.get(CAMPO_VALOR),
                "situacao_resultado":     r.get(CAMPO_SITUACAO),
            })
    return registros

print("Funções definidas.")

Funções definidas.


### 4. Teste com amostra (5 contratações)

In [13]:
amostra = df_com_resultado.head(5)
registros_teste = []

for _, row in amostra.iterrows():
    recs = extrair_resultados_contratacao(
        cnpj=row["cnpj_norm"],
        ano=row["anoCompraPncp"],
        seq=row["sequencialCompraPncp"],
        id_compra=row["idCompra"]
    )
    registros_teste.extend(recs)
    print(f"  idCompra={row['idCompra']}: {len(recs)} resultados")
    time.sleep(0.5)

df_teste = pd.DataFrame(registros_teste)
print(f"\nTotal: {len(df_teste)} linhas")
print(df_teste.head(3))

  idCompra=38919505000012023: 2 resultados
  idCompra=15518005000572022: 45 resultados
  idCompra=15518005000022023: 18 resultados
  idCompra=15518005000322022: 1 resultados
  idCompra=38919505000022023: 6 resultados

Total: 72 linhas
            idCompra orgaoEntidadeCnpj anoCompraPncp sequencialCompraPncp  \
0  38919505000012023    09791450000114          2023                    1   
1  38919505000012023    09791450000114          2023                    1   
2  15518005000572022    15126437000143          2022                  842   

   numeroItem                                     descricaoItem  \
0           1  Prestaçao de Serviços de Agenciamento de Viagens   
1           2  Prestaçao de Serviços de Agenciamento de Viagens   
2           1          Parafuso Ósseo - Mini E Micro Fragmentos   

  cnpj_fornecedor            nome_fornecedor  valor_total_homologado  \
0  07832586000108  DF TURISMO E EVENTOS LTDA                     0.6   
1  07832586000108  DF TURISMO E EVENTOS LTD

### 5. Extração completa — todas as contratações com resultado

Descomente após validar a amostra acima. Checkpoint a cada 100 contratações.

In [ ]:
checkpoint_file = "data/raw/propostas_pe.csv"
MAX_WORKERS = 20  

if os.path.exists(checkpoint_file):
    df_existente = pd.read_csv(checkpoint_file, dtype=str)
    ids_prontos  = set(df_existente["idCompra"].unique())
    todos_registros = df_existente.to_dict("records")
    print(f"Retomando: {len(ids_prontos)} contratações já processadas, {len(todos_registros)} registros")
else:
    ids_prontos = set()
    todos_registros = []

pendentes = df_com_resultado[~df_com_resultado["idCompra"].isin(ids_prontos)]
print(f"{len(pendentes)}/{len(df_com_resultado)} contratações a processar")

lock    = threading.Lock()
counter = [0]  

def processar_row(row):
    return extrair_resultados_contratacao(
        cnpj=row["cnpj_norm"],
        ano=row["anoCompraPncp"],
        seq=row["sequencialCompraPncp"],
        id_compra=row["idCompra"]
    )

rows = [row for _, row in pendentes.iterrows()]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futuros = {executor.submit(processar_row, row): row["idCompra"] for row in rows}
    for futuro in as_completed(futuros):
        recs = futuro.result() or []
        with lock:
            todos_registros.extend(recs)
            counter[0] += 1
            if counter[0] % 100 == 0:
                pd.DataFrame(todos_registros).to_csv(checkpoint_file, index=False)
                print(f"  {counter[0]}/{len(pendentes)} | {len(todos_registros)} registros")

propostas_pe = pd.DataFrame(todos_registros)
propostas_pe.to_csv(checkpoint_file, index=False)
print(f"\n{len(propostas_pe)} registros salvos em {checkpoint_file}")
print(f"CNPJs únicos de fornecedores: {propostas_pe['cnpj_fornecedor'].nunique()}")

Retomando: 1095 contratações já processadas, 6551 registros
8958/10057 contratações a processar
  100/8958 | 7439 registros
  200/8958 | 8189 registros
  300/8958 | 8745 registros
  400/8958 | 10384 registros
  500/8958 | 11774 registros
  600/8958 | 12102 registros
  700/8958 | 12934 registros
  800/8958 | 13501 registros
  900/8958 | 13892 registros
  1000/8958 | 14574 registros
  1100/8958 | 15686 registros
  1200/8958 | 16130 registros
  1300/8958 | 17357 registros
  1400/8958 | 18678 registros
  1500/8958 | 19238 registros
  1600/8958 | 21064 registros
  1700/8958 | 22151 registros
  1800/8958 | 23012 registros
  1900/8958 | 24064 registros
  2000/8958 | 25341 registros
  2100/8958 | 26895 registros
  2200/8958 | 27862 registros
  2300/8958 | 28093 registros
  2400/8958 | 28372 registros
  2500/8958 | 31202 registros
  2600/8958 | 31680 registros
  2700/8958 | 32219 registros
  2800/8958 | 32535 registros
  2900/8958 | 32636 registros
  3000/8958 | 32738 registros
  3100/8958 | 32

### 6. Patch — re-busca contratações com campos nulos (checkpoint antigo pré-correção)

In [ ]:
df_atual = pd.read_csv(checkpoint_file, dtype=str)

null_mask = df_atual["valor_total_homologado"].isna()
ids_nulos = set(df_atual.loc[null_mask, "idCompra"].unique())
print(f"Registros com campos nulos: {null_mask.sum()} ({len(ids_nulos)} contratações únicas)")

df_clean = df_atual[~null_mask].copy()
print(f"Registros que permanecem (campos OK): {len(df_clean)}")

df_para_refetch = df_com_resultado[df_com_resultado["idCompra"].isin(ids_nulos)].copy()
print(f"Contratações a re-buscar: {len(df_para_refetch)}")

rows_refetch = [row for _, row in df_para_refetch.iterrows()]
novos_registros = []
counter_rf = [0]
lock_rf = threading.Lock()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futuros_rf = {executor.submit(processar_row, row): row["idCompra"] for row in rows_refetch}
    for futuro in as_completed(futuros_rf):
        recs = futuro.result() or []
        with lock_rf:
            novos_registros.extend(recs)
            counter_rf[0] += 1
            if counter_rf[0] % 100 == 0:
                print(f"  {counter_rf[0]}/{len(rows_refetch)} | {len(novos_registros)} novos registros")

print(f"\nNovos registros coletados: {len(novos_registros)}")

df_patch = pd.concat([df_clean, pd.DataFrame(novos_registros)], ignore_index=True)
df_patch.to_csv(checkpoint_file, index=False)
print(f"Total final: {len(df_patch)} registros em {checkpoint_file}")

still_null = df_patch["valor_total_homologado"].isna().sum()
print(f"Campos nulos remanescentes: {still_null}")

Registros com campos nulos: 6551 (1095 contratações únicas)
Registros que permanecem (campos OK): 68949
Contratações a re-buscar: 1099
  100/1099 | 175 novos registros
  200/1099 | 450 novos registros
  300/1099 | 751 novos registros
  400/1099 | 1133 novos registros
  500/1099 | 1691 novos registros
  600/1099 | 2333 novos registros
  700/1099 | 4163 novos registros
  800/1099 | 4524 novos registros
  900/1099 | 4728 novos registros
  1000/1099 | 5401 novos registros

Novos registros coletados: 6553
Total final: 75502 registros em data/raw/propostas_pe.csv
Campos nulos remanescentes: 0
